In [5]:
import nibabel as nib
import os
import matplotlib.pyplot as plt
import numpy as np
import pydicom
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, VBox, Output, Label, interactive
from IPython.display import display, clear_output
from datetime import datetime



from utils.dcm2niix import *

# Project data path
ANALYSIS_PATH='/home/bule/berner_daten_analyse/data'
APP_DATA_PATH = '/home/bule/berner_daten_analyse/apps/app_manual_preselection'
NII_TARGET_PATH = '/home/bule/berner_daten_analyse/data/nifti_pascal_selection_try'
LOG_PATH = os.path.join(NII_TARGET_PATH, 'conversion_errors_pascal_selection.log')


In [7]:
dwi_t1_df_2=pd.read_csv(os.path.join(ANALYSIS_PATH,"dwi_t1_df_2.csv"))
dwi_t1_df_2.sort_values(by='Patient_ID',inplace=True)
dwi_t1_df_2.head()
len(dwi_t1_df_2)

dwi_t1_df_2.groupby('Patient_ID').nunique()

,Study_ID,Sequence_ID,File_Name,Full_Path,Date,Sequence_ID_split,Modality
Patient_ID,,,,,,,
76813,3,4,50,167,3,3,2
442968,2,2,38,76,2,1,1
516341,2,5,160,302,2,3,2
607681,2,3,176,230,2,3,1
653853,4,6,261,455,4,4,2
...,...,...,...,...,...,...,...
82298262,3,5,38,162,3,2,2
82341206,3,5,50,172,3,3,2
82364672,1,3,50,98,1,2,2


In [8]:
#dwi
df_dwi = dwi_t1_df_2[dwi_t1_df_2['Modality']=='DWI']

source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in df_dwi['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])

target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences_dwi= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences_dwi[0])

# t1
df_t1= dwi_t1_df_2[dwi_t1_df_2['Modality']=='T1']
source_dcm_sequences = [os.path.join('/',*path.split('/')[:-1]) for path in df_t1['Full_Path']]
source_dcm_sequences = list(set([os.path.join('/',path) for path in source_dcm_sequences]))
print(source_dcm_sequences[0])

target_nii_sequences= [os.path.join(*path.split('/')[4:]) for path in source_dcm_sequences]
target_nii_sequences_t1= [os.path.join(NII_TARGET_PATH,path) for path in target_nii_sequences]
print(target_nii_sequences_t1[0])

/nasdata/Berner_Daten/Other_Side_Bilddaten/0012089796/20140410_3383245/ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30.27272.2014041015375934999471229.0.0.0
/home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0012089796/20140410_3383245/ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30.27272.2014041015375934999471229.0.0.0
/nasdata/Berner_Daten/Other_Side_Bilddaten/0005559340/20160122_3788504/t1_fl2d_tra__1.3.12.2.1107.5.2.36.40186.2016012212035612876170035.0.0.0
/home/bule/berner_daten_analyse/data/nifti_pascal_selection_try/0005559340/20160122_3788504/t1_fl2d_tra__1.3.12.2.1107.5.2.36.40186.2016012212035612876170035.0.0.0


In [13]:
import json
with open(os.path.join(APP_DATA_PATH,'pascal_sequences.json')) as json_file:
    selection_pascal_dict = json.load(json_file)

rows = []
for patid, selection_dict in selection_pascal_dict.items():
    for modality, sequences in selection_dict.items():
        for sequence in sequences:
            study_id, sequence_id = sequence.split('/')
            rows.append({
                'Patient_ID': patid,
                'Study_ID': study_id,
                'Sequence_ID': sequence_id,
                'Modality': modality,
                'Labeller': 'Pascal'  
            })
selection_pascal_df = pd.DataFrame(rows)


dwi_df=selection_pascal_df[selection_pascal_df['Modality']=='dwi'].sort_values(by='Patient_ID')
dwi_df['Sequence_path']=dwi_df.apply(lambda x: os.path.join(ANALYSIS_PATH,x['Patient_ID'],x['Study_ID'],x['Sequence_ID']),axis=1)
t1_df=selection_pascal_df[selection_pascal_df['Modality']=='t1'].sort_values(by='Patient_ID')
t1_df['Sequence_path']=t1_df.apply(lambda x: os.path.join(ANALYSIS_PATH,x['Patient_ID'],x['Study_ID'],x['Sequence_ID']),axis=1)


[0]


all_niftis=list_nifti_files(NII_TARGET_PATH)
df_modality = pd.DataFrame({
    'Path_Nifti': list(dwi_df['Sequence_path'])+list(t1_df['Sequence_path']),
    'Modality': ['dwi'] * len(list(dwi_df['Sequence_path']))+['t1'] * len(list(t1_df['Sequence_path'])),
})
df_modality['join_crit']=[ os.path.join('/',*path.split('/')[6:]) for path in df_modality['Path_Nifti']] # just  to get the modality
df_modality.head()

,Path_Nifti,Modality,join_crit
0,/home/bule/berner_daten_analyse/data/000007681...,dwi,/20170614_4361718/ep2d_diff_M128_b0_1000_DIN__...
1,/home/bule/berner_daten_analyse/data/000044296...,dwi,/20151109_3835318/ep2d_diff_3scan_trace_p2__1....
2,/home/bule/berner_daten_analyse/data/000048185...,dwi,/20121025_3013771/resolve_3scan_trace_tra_176_...
3,/home/bule/berner_daten_analyse/data/000051634...,dwi,/20160211_3914130/ep2d_diff_M128_b0_1000_DIN__...
4,/home/bule/berner_daten_analyse/data/000060768...,dwi,/20170707_4385986/resolve_4scan_trace_tra_diff...


In [2]:
all_niftis=list_nifti_files(NII_TARGET_PATH)
df_modality = pd.DataFrame({
    'Path_Nifti': target_nii_sequences_dwi+target_nii_sequences_t1,
    'Modality': ['dwi'] * len(target_nii_sequences_dwi)+['t1'] * len(target_nii_sequences_t1),
})
df_modality['join_crit']=[ os.path.join('/',*path.split('/')[6:]) for path in df_modality['Path_Nifti']] # just  to get the modality
df_modality.head()

NameError: name 'target_nii_sequences_dwi' is not defined

In [14]:
data=[]
for path in all_niftis:
    patient_id = int(path.split("/")[6])
    study_id = path.split("/")[7]
    sequence_id = path.split("/")[8]
    full_path = path
    data.append({'Patient_ID':patient_id, 'Study_ID':study_id,'Sequence_ID':sequence_id ,'Full_Path_Nifti':full_path})
    
nii_df=pd.DataFrame(data)
nii_df['join_crit']=[ os.path.join('/',*path.split('/')[6:-1]) for path in nii_df['Full_Path_Nifti']]

nii_df=nii_df.merge(df_modality, on='join_crit', how='left')

# len(nii_df[['Sequence_ID','Study_ID']].unique()) 
duplicates = nii_df[nii_df.duplicated(subset=['Study_ID', 'Sequence_ID'], keep=False)]
duplicates

,Patient_ID,Study_ID,Sequence_ID,Full_Path_Nifti,join_crit,Path_Nifti,Modality


- somehow there  are some duplicatet t1s 

In [15]:
nii_df.drop_duplicates(subset=['Sequence_ID'], keep='first', inplace=True)
nii_df

,Patient_ID,Study_ID,Sequence_ID,Full_Path_Nifti,join_crit,Path_Nifti,Modality
0,1029355,20150421_3667803,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001029355/20150421_3667803/ep2d_diff_3scan_t...,NaN,NaN
1,1029355,20150421_3667803,t1_se_r_tra_KM__1.3.12.2.1107.5.2.30.27272.201...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001029355/20150421_3667803/t1_se_r_tra_KM__1...,NaN,NaN
2,3082016,20150215_3615242,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0003082016/20150215_3615242/ep2d_diff_3scan_t...,NaN,NaN
3,3082016,20150215_3615242,t1_se_tra__1.3.12.2.1107.5.2.30.27272.20150215...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0003082016/20150215_3615242/t1_se_tra__1.3.12...,NaN,NaN
4,2782782,20160119_3893609,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,/home/bule/berner_daten_analyse/data/nifti_pas...,/0002782782/20160119_3893609/ep2d_diff_M128_b0...,NaN,NaN
...,...,...,...,...,...,...,...
244,2593122,20160922_4112749,t1_tse_r_tra__1.3.12.2.1107.5.2.18.41437.20160...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0002593122/20160922_4112749/t1_tse_r_tra__1.3...,NaN,NaN
245,1489860,20120330_2884717,t1_se_r_tra_KM__1.3.12.2.1107.5.2.30.27272.201...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001489860/20120330_2884717/t1_se_r_tra_KM__1...,NaN,NaN
246,1489860,20120330_2884717,ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001489860/20120330_2884717/ep2d_diff_3scan_t...,NaN,NaN
247,1424025,20170712_4389337,ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2....,/home/bule/berner_daten_analyse/data/nifti_pas...,/0001424025/20170712_4389337/ep2d_diff_M128_b0...,NaN,NaN


- OLEA analysis seems to make problems here -> removed them 

In [ ]:
file_path='/home/bule/berner_daten_analyse/data/nifti_second_try/0000076813/20170614_4361718/ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2.36.40186.2017061416530865378051255.0.0.0/ep2d_diff_M128_b0_1000_DIN_1.3.12.2.1107.5.2.36.40186.2017061416530865378051255.0.0.0_ep2d_diff_M128_b0_1000_DIN_20170614164701_3.nii.gz'
img = nib.load(file_path)
data_dwi = img.get_fdata()
def view_slice_dwi(slice_idx):
    clear_output(wait=True)  # Clear previous output to avoid duplicate plots
    plt.figure(figsize=(3, 3))
    plt.imshow(data[:, :, slice_idx,1].squeeze(), cmap="gray", origin="lower")
    plt.title(f"Slice {slice_idx}")
    plt.axis("off")
    plt.show()
slider_dwi = IntSlider(min=0, max=data_dwi.shape[2] - 1, step=1, value=0)
interactive_plot_dwi = interactive(view_slice_dwi, slice_idx=slider_dwi)
display(interactive_plot_dwi)

In [ ]:
file_path='/home/bule/berner_daten_analyse/data/nifti_second_try/0000076813/20170929_4371316/t1_se_tra__1.3.12.2.1107.5.2.41.169589.2017092912262239073557517.0.0.0/t1_se_tra_1.3.12.2.1107.5.2.41.169589.2017092912262239073557517.0.0.0_t1_se_tra_20170929120431_19.nii.gz'
img = nib.load(file_path)
data = img.get_fdata()
def view_slice(slice_idx):
    clear_output(wait=True)  # Clear previous output to avoid duplicate plots
    plt.figure(figsize=(3, 3))
    plt.imshow(data[:, :, slice_idx].squeeze(), cmap="gray", origin="lower")
    plt.title(f"Slice {slice_idx}")
    plt.axis("off")
    plt.show()
    
slider = IntSlider(min=0, max=data.shape[2] - 1, step=1, value=0)
interactive_plot = interactive(view_slice, slice_idx=slider)
display(interactive_plot)

In [ ]:
df = nii_df.sort_values(by='Patient_ID')

df['Date'] = df['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))

pd.set_option('display.max_colwidth', None)

patient_dropdown = Dropdown(options=df['Patient_ID'].unique(), description='Patient:')
sequence_dropdown = Dropdown(description='Sequence:')
sequences_output = Output()
slider_output = Output()  
plot_output = Output()    

def update_sequences(*args):
    patient_id = patient_dropdown.value
    patient_data = df[df['Patient_ID'] == patient_id]
    sequence_dropdown.options = patient_data['Sequence_ID'].values

    with sequences_output:
        clear_output(wait=True)
        display(patient_data[['Date', 'Sequence_ID', 'Modality']].sort_values(by='Date'))

def display_modality(*args):
    patient_id = patient_dropdown.value
    sequence_id = sequence_dropdown.value
    if sequence_id is None:
        return
    row = df[(df['Patient_ID'] == patient_id) & (df['Sequence_ID'] == sequence_id)].iloc[0]
    file_path = row['Full_Path_Nifti']
    modality = row['Modality']

    img = nib.load(file_path)
    data = img.get_fdata()
    if data.ndim == 4:
        data=data[:,:,:,1]
        
    def view_slice(slice_idx):
        with plot_output:
            clear_output(wait=True)
            plt.figure(figsize=(5, 5))
            plt.imshow(data[:, :, slice_idx].squeeze(), cmap="gray", origin="lower")
            plt.title(f"Slice {slice_idx}")
            plt.axis("off")
            plt.show()
    slider = IntSlider(min=0, max=data.shape[2] - 1, step=1, value=0)
    slider.observe(lambda change: view_slice(slider.value), names='value')

    with slider_output:
        clear_output(wait=True)
        display(slider)
    view_slice(0)

patient_dropdown.observe(update_sequences, names='value')
sequence_dropdown.observe(display_modality, names='value')

update_sequences()

display(VBox([
    patient_dropdown,
    sequences_output,
    sequence_dropdown,
    slider_output,
    plot_output
]))




In [ ]:
!pip3 install torch torchvision torchaudio

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
df.groupby('Patient_ID').nunique()

In [ ]:
def select_modalities(patient_data):
    dwi = patient_data[patient_data['Modality'] == 'dwi'].sort_values(by='Date').head(1)
    t1 = patient_data[patient_data['Modality'] == 't1'].sort_values(by='Date').head(1)
    if not dwi.empty and not t1.empty:
        return pd.concat([dwi, t1])
    return pd.DataFrame()  

only_two_df = df.groupby('Patient_ID', group_keys=False).apply(select_modalities)
only_two_df.reset_index(drop=True, inplace=True)
only_two_df.groupby('Patient_ID').nunique()

In [89]:
only_two_df.to_csv(os.path.join(ANALYSIS_PATH, 'only2_df.csv'), index=False)

In [ ]:
df

### Compare to Dicom


- seems to fit

In [ ]:
dicom_data = pydicom.dcmread('/nasdata/Berner_Daten/Other_Side_Bilddaten/0000076813/20170614_4361718/ep2d_diff_M128_b0_1000_DIN__1.3.12.2.1107.5.2.36.40186.2017061416530865378051255.0.0.0/15.dcm'
, )  # Load without pixel data to check header only
dicom_data.pixel_array.shape

plt.imshow(dicom_data.pixel_array, cmap="gray", origin="lower")

In [ ]:


dicom_data = pydicom.dcmread('/nasdata/Berner_Daten/Other_Side_Bilddaten/0013140710/20160101_3878363/B1000_OLEA_ANALYSIS__1.3.6.1.4.1.32839.99.2.228328515211608092063513064401162864980/2.dcm'
, )  # Load without pixel data to check header only
dicom_data.pixel_array.shape

plt.imshow(dicom_data.pixel_array, cmap="gray", origin="lower")